This script builds a master inventory (“data catalogue”) of what MRI + MEG files exist on disk, per subject and session, and writes a single CSV ('master_data_catalogue.csv') that downstream scripts will use as the authoritative roster of what data is available.

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, re
import datetime
import pandas as pd
from pathlib import Path
import shutil
import numpy as np

In [ ]:
### SET PARAMETERS:

MRI_root = config['MRI_data_directory']
MEG_root = config['MEG_data_directory']

base_output_directory = config['root_output_directory']

HARD_STOP = config['hard_errors']

SESSION_MAPPINGS = config['session_ID_mappings']
DATATYPE_MAPPINGS = config.get("datatype_mappings", {})
GROUP_MAPPINGS = config.get("group_identifiers") or {}
STRIP_SUBJECTID_SUBSTRINGS = config.get("strip_subjectID_substrings", None)

# Display loaded paths and parameters from config file:
print(f"--> Cataloging MRI data from path: {MRI_root}")
print(f"--> Cataloging MEG data from path: {MEG_root}")

if HARD_STOP == True:
    print("\n--> Error mode: 'STRICT' (hard errors will be thrown if unexpected/incompatible data are detected)")
elif HARD_STOP == False:
    print("\n--> Error mode: 'SOFT' (only printed warnings will be thrown, and subject_IDs with unexpected data will be dropped from data catalogue)")
else:
    print("\nUnexpected error mode parameter setting: proceeding in 'STRICT' mode")
    HARD_STOP = True

print(f"\nOutput data catalogue will be stored in: {base_output_directory}")

In [ ]:
# =========================
# Catalog (MRI + MEG) into a single dataframe
# =========================

# Safety checks on roots
for label, root in [("MRI_root", MRI_root), ("MEG_root", MEG_root)]:
    if not Path(root).exists():
        raise FileNotFoundError(f"{label} does not exist: {root}")

# Normalize roots to Path objects
MRI_root_path = Path(MRI_root)
MEG_root_path = Path(MEG_root)

rows = []

# -------------------------
# Pass 1: MRI (NIfTI)
# -------------------------
for root_directory, subdirectories, filenames in os.walk(MRI_root_path):
    root_path = Path(root_directory)

    for filename in filenames:
        file_path = root_path / filename

        # Handle .nii and .nii.gz
        full_suffix = "".join(file_path.suffixes)
        if full_suffix == ".nii.gz":
            extension = ".nii.gz"
        else:
            extension = file_path.suffix

        if extension not in {".nii", ".nii.gz"}:
            continue

        # -------------------------
        # MRI parsing (BIDS-ish):
        # - REQUIRE: sub-<subject>
        # - OPTIONAL: ses-<session>
        # - If ses- missing: assign UNKNOWN (disambiguated later as UNKNOWN_2, UNKNOWN_3, ...)
        # -------------------------
        filename_string = file_path.name
        path_string = str(file_path)

        # Find sub- token in filename OR anywhere in path (required)
        m_sub = re.search(r"(sub-[A-Za-z0-9]+)", filename_string)
        if m_sub is None:
            m_sub = re.search(r"(sub-[A-Za-z0-9]+)", path_string)

        if m_sub is None:
            message = f"Could not find required 'sub-' token in MRI path/filename: {file_path}"
            if HARD_STOP:
                raise ValueError(message)
            else:
                print("[warn]", message)
                continue

        subject_token = m_sub.group(1)  # e.g. "sub-001"
        subject_id = subject_token.replace("sub-", "")

        # Find ses- token in filename OR path (optional)
        m_ses = re.search(r"(ses-[A-Za-z0-9]+)", filename_string)
        if m_ses is None:
            m_ses = re.search(r"(ses-[A-Za-z0-9]+)", path_string)

        if m_ses is not None:
            raw_session_label = m_ses.group(1).replace("ses-", "")
        else:
            raw_session_label = None

        session_label = raw_session_label

        # Apply optional session_ID_mappings (always overriding if they match)
        if SESSION_MAPPINGS:
            mapping_matches = []
            for mapping_label, substring_list in SESSION_MAPPINGS.items():
                if substring_list is None:
                    continue
                for substring in substring_list:
                    if substring in filename_string:
                        mapping_matches.append(mapping_label)
                        break

            if len(mapping_matches) == 1:
                session_label = mapping_matches[0]
            elif len(mapping_matches) > 1:
                message = f"Multiple session_ID_mappings matched MRI file {file_path}: {mapping_matches}"
                if HARD_STOP:
                    raise ValueError(message)
                else:
                    print("[warn]", message)
                    session_label = mapping_matches[0]
            else:
                session_label = raw_session_label
        else:
            session_label = raw_session_label

        if session_label is None:
            session_label = "UNKNOWN"

        rows.append({
            "path":           str(file_path),
            "BIDS_extension": extension,
            "BIDS_datatype":  "anat",
            "BIDS_suffix":    "T1w",      # treat these as T1w structurals
            "BIDS_subject":   subject_id,  # stored without 'sub-' for backward compatibility; participant_id adds it later
            "BIDS_session":   session_label,
            "BIDS_task":      np.nan,
            "BIDS_run":       np.nan,
            "BIDS_acq":       np.nan,
            "BIDS_dir":       np.nan,
            "BIDS_echo":      np.nan,
            "BIDS_space":     np.nan,
            "BIDS_desc":      np.nan})

# -------------------------
# Pass 2: MEG (FIF)
# -------------------------
for root_directory, subdirectories, filenames in os.walk(MEG_root_path):
    root_path = Path(root_directory)

    for filename in filenames:
        file_path = root_path / filename

        if file_path.suffix != ".fif":
            continue
        if "_meg" not in file_path.name:
            continue

        # Strip extension
        filename_without_extension = file_path.name[:-len(".fif")]
        name_parts = filename_without_extension.split("_")

        subject_token = None
        session_token = None
        task_token = None
        run_token = None

        for token in name_parts:
            if token.startswith("sub-"):
                subject_token = token
            elif token.startswith("ses-"):
                session_token = token
            elif token.startswith("task-"):
                task_token = token
            elif token.startswith("run-"):
                run_token = token

        if subject_token is None:
            message = f"Could not parse subject_ID from MEG filename: {file_path.name}"
            if HARD_STOP:
                raise ValueError(message)
            else:
                print("[warn]", message)
                continue

        subject_id = subject_token.replace("sub-", "")

        if session_token is not None:
            raw_session_label = session_token.replace("ses-", "")
        else:
            raw_session_label = None

        session_label = raw_session_label

        # Apply optional session_ID_mappings (override any ses- token)
        filename_string = file_path.name
        if SESSION_MAPPINGS:
            mapping_matches = []
            for mapping_label, substring_list in SESSION_MAPPINGS.items():
                if substring_list is None:
                    continue
                for substring in substring_list:
                    if substring in filename_string:
                        mapping_matches.append(mapping_label)
                        break

            if len(mapping_matches) == 1:
                session_label = mapping_matches[0]
            elif len(mapping_matches) > 1:
                message = f"Multiple session_ID_mappings matched MEG file {file_path}: {mapping_matches}"
                if HARD_STOP:
                    raise ValueError(message)
                else:
                    print("[warn]", message)
                    session_label = mapping_matches[0]
            # if there are no matches, session_label stays as raw_session_label

        if session_label is None:
            session_label = "UNKNOWN"

        if task_token is not None:
            task_label = task_token.replace("task-", "")
        else:
            task_label = np.nan

        if run_token is not None:
            run_label = run_token.replace("run-", "")
        else:
            run_label = np.nan

        rows.append({
            "path":           str(file_path),
            "BIDS_extension": ".fif",
            "BIDS_datatype":  "meg",
            "BIDS_suffix":    "meg",
            "BIDS_subject":   subject_id,
            "BIDS_session":   session_label,
            "BIDS_task":      task_label,
            "BIDS_run":       run_label,
            "BIDS_acq":       np.nan,
            "BIDS_dir":       np.nan,
            "BIDS_echo":      np.nan,
            "BIDS_space":     np.nan,
            "BIDS_desc":      np.nan})

# -------------------------
# Build dataframe
# -------------------------
df = pd.DataFrame(rows)

if df.empty:
    print("[info] No MRI/MEG files found under the provided roots.")
else:
    # Sort into a stable, human-friendly order (if columns exist)
    sort_columns = [column for column in [
        "BIDS_datatype",
        "BIDS_suffix",
        "BIDS_subject",
        "BIDS_session",
        "BIDS_task",
        "BIDS_run",
        "BIDS_acq",
        "BIDS_dir",
        "BIDS_echo",
        "BIDS_space",
        "BIDS_desc",
        "BIDS_extension"
    ] if column in df.columns]

    if sort_columns:
        df = df.sort_values(sort_columns, kind="stable").reset_index(drop=True)

    # -------------------------
    # Disambiguate MRI sessions when ses- is missing:
    # For each subject with multiple MRI rows where BIDS_session == "UNKNOWN",
    # assign "UNKNOWN", "UNKNOWN-2", "UNKNOWN-3", ...
    # -------------------------
    mask_mri_unknown = (df["BIDS_datatype"] == "anat") & (df["BIDS_session"].astype(str) == "UNKNOWN")
    if mask_mri_unknown.any():
        df = df.copy()
        df["_tmp_order"] = df["path"].astype(str)

        def _relabel_unknown_sessions(group: pd.DataFrame) -> pd.DataFrame:
            group = group.sort_values("_tmp_order", kind="stable").copy()
            unknown_mask = (group["BIDS_datatype"] == "anat") & (group["BIDS_session"].astype(str) == "UNKNOWN")
            if int(unknown_mask.sum()) <= 1:
                return group

            unknown_indices = group.index[unknown_mask].tolist()
            for k, idx in enumerate(unknown_indices, start=1):
                group.at[idx, "BIDS_session"] = "UNKNOWN" if k == 1 else f"UNKNOWN-{k}"
            return group

        df = (
            df.groupby("BIDS_subject", group_keys=False)
              .apply(_relabel_unknown_sessions)
              .drop(columns=["_tmp_order"])
              .reset_index(drop=True))

    # ========== participants.tsv export (multi-modality) ==========
    if "BIDS_subject" in df.columns and df["BIDS_subject"].notna().any():
        participants_dataframe = (
            df.dropna(subset=["BIDS_subject", "BIDS_suffix", "BIDS_session"])
            .assign(
                participant_id=lambda x: x["BIDS_subject"].astype(str).apply(
                    lambda s: s if s.startswith("sub-") else f"sub-{s}"))
            .groupby(["participant_id", "BIDS_suffix"], as_index=False)
            .agg({"BIDS_session": lambda series: ",".join(sorted(set(series.astype(str))))})
            .pivot(index="participant_id", columns="BIDS_suffix", values="BIDS_session")
            .reset_index()
            .rename_axis(None, axis=1))

        participants_dataframe = participants_dataframe.rename(
            columns={column: f"{column}_sessions" for column in participants_dataframe.columns
                     if column != "participant_id"})
        participants_dataframe = participants_dataframe.fillna("n/a")
        out_path_participants = Path(base_output_directory) / "participants.tsv"
        participants_dataframe.to_csv(out_path_participants, sep="\t", index=False, na_rep="n/a")
        print(f"[write] participants.tsv → {out_path_participants}  ({len(participants_dataframe)} rows)")
    else:
        print("[info] No BIDS_subject values found; skipping participants.tsv export.")

    # ========== Column pruning & diagnostics ==========
    if df.empty:
        print("[info] df is empty; skipping column pruning.")
    else:
        # 1) Drop fully-empty columns (all NaN)
        all_null_columns = [column for column in df.columns if df[column].isna().all()]
        if all_null_columns:
            print("[drop: no data] Columns dropped because they contain only NA/empty values:")
            for column in all_null_columns:
                print(f"   - {column}")
            df = df.drop(columns=all_null_columns)
        else:
            print("[drop: no data] None")

        # 2) Report & drop single-valued (uninformative) columns
        exclusion_list = ["BIDS_suffix", "BIDS_subject", "BIDS_session", "BIDS_task"]
        always_keep = {"path"}  # never drop

        single_valued_columns = []

        for column in df.columns:
            if column in always_keep:
                continue

            series_non_null = df[column].dropna()
            temporary_values = []

            for value in series_non_null:
                if isinstance(value, (list, tuple, np.ndarray)):
                    temporary_values.append(tuple(value))
                elif isinstance(value, dict):
                    temporary_values.append(tuple(sorted(value.items())))
                else:
                    try:
                        hash(value)
                        temporary_values.append(value)
                    except TypeError:
                        temporary_values.append(repr(value))

            unique_values = pd.unique(pd.Series(temporary_values))
            if len(unique_values) == 1 and column not in exclusion_list:
                single_valued_columns.append((column, unique_values[0]))

        if single_valued_columns:
            print("[drop: single-valued] Columns dropped because they had a single value across all rows:")
            for column_name, value in single_valued_columns:
                print(f"   - {column_name}: {value!r}")
            df = df.drop(columns=[column_name for column_name, _ in single_valued_columns])
        else:
            print("[drop: single-valued] None")

        # 3) Final compact summary
        print(f"[result] df shape: {df.shape[0]} rows × {df.shape[1]} cols")
        df.head()

In [ ]:
# =========================
# Session ID summary (no remapping; BIDS_session is already canonical)
# =========================

if "BIDS_session" not in df.columns:
    raise KeyError("Column 'BIDS_session' not found in df.")

# Just alias BIDS_session -> session_ID for compatibility with older code
df["session_ID"] = df["BIDS_session"]

# Print counts in YAML-defined order if mappings are present
value_counts = df["session_ID"].value_counts(dropna=True)

if SESSION_MAPPINGS:
    print("[info] Session counts (following order in session_ID_mappings):")
    for label in SESSION_MAPPINGS.keys():
        print(f"\t{label} sessions:\t{int(value_counts.get(label, 0))}")
else:
    print("[info] Session counts (all observed labels):")
    for label, count in value_counts.items():
        print(f"\t{label} sessions:\t{int(count)}")

# Also report rows where session_ID is NaN
n_missing_sessions = int(df["session_ID"].isna().sum())
if n_missing_sessions:
    print(f"[info] rows without session_ID (NaN): {n_missing_sessions}")

In [ ]:
# =========================
# Data type mapping (config-driven): BIDS_suffix -> data_type (e.g., MRI / fMRI)
# =========================

# Display mappings (destination keys = data_type labels; values = lists of BIDS_suffix codes)
print("[info] datatype_mappings:", DATATYPE_MAPPINGS)

if "BIDS_suffix" not in df.columns:
    raise KeyError("Column 'BIDS_suffix' not found in df.")

# Build reverse lookup: suffix -> data_type (e.g., "T1w" -> "MRI", "bold" -> "fMRI"), checking for conflicts
suffix_to_type = {}
for dtype_label, suffixes in (DATATYPE_MAPPINGS or {}).items():
    if suffixes is None:
        continue
    for suf in suffixes:
        key = str(suf).strip()
        if key in suffix_to_type and suffix_to_type[key] != dtype_label:
            raise ValueError(f"Conflicting mapping for suffix '{key}': '{suffix_to_type[key]}' vs '{dtype_label}'")
        suffix_to_type[key] = dtype_label

# 1) Validate all observed suffixes are mappable (ignore NaNs)
observed_suffixes = pd.Series(df["BIDS_suffix"]).dropna().astype(str).str.strip().unique()
unknown_suffixes = sorted([s for s in observed_suffixes if s not in suffix_to_type])

if unknown_suffixes:
    msg = "[data_type mapping] Unmapped BIDS_suffix value(s) encountered: " + ", ".join(unknown_suffixes)
    if HARD_STOP:
        # Hard stop mode
        raise ValueError(msg + ". Please add these under 'datatype_mappings' in config.yaml.")
    else:
        # Soft mode: warn and drop only offending rows (not whole subjects)
        print("[warn]", msg)
        print("[warn] Dropping rows with these unmapped suffixes (rows only, not entire subjects).")
        df = df[~df["BIDS_suffix"].astype(str).isin(unknown_suffixes)].reset_index(drop=True)
else:
    print("[info] All BIDS_suffix values successfully mapped.")

# 2) Create the new 'data_type' column
df["data_type"] = df["BIDS_suffix"].apply(
    lambda x: suffix_to_type.get(str(x).strip()) if pd.notna(x) else np.nan)

# 3) Print concise counts per data_type (respect order as in config)
counts = df["data_type"].value_counts(dropna=True)
for dtype_label in DATATYPE_MAPPINGS.keys():
    print(f"\t{dtype_label} files found: {int(counts.get(dtype_label, 0))}")

# Optional: report rows without a data_type (i.e., BIDS_suffix was NaN)
n_missing_dtype = int(df["data_type"].isna().sum())
if n_missing_dtype:
    print(f"[info] rows without data_type (NaN): {n_missing_dtype}")


In [ ]:
# =========================
# Group assignment from path substrings (config-driven)
# =========================

if "path" not in df.columns:
    raise KeyError("Column 'path' not found in df.")
if "BIDS_subject" not in df.columns:
    raise KeyError("Column 'BIDS_subject' not found in df.")

# 1) Create 'group_ID' for all rows
if not GROUP_MAPPINGS:  # empty dict or missing / None
    print("[warn] No 'group_identifiers' found in config.yaml; assigning group_ID='UNKNOWN' for all rows.")
    df["group_ID"] = "UNKNOWN"
else:
    # Build a normalized, case-insensitive search over path
    df["_path_lc"] = df["path"].astype(str).str.lower()

    # Prepare storage
    assigned = []        # single chosen group key per row (or None)
    matched_multi = []   # number of matched groups per row (for diagnostics)
    matched_groups_col = []  # list of matched group keys per row

    # Iterate rows and find matching group keys by substring (case-insensitive)
    for i in range(len(df)):
        path_lowercase = df["_path_lc"].iloc[i]
        hits = []

        # loop over mapping keys and their substrings
        for group_key, substrings in GROUP_MAPPINGS.items():
            if substrings is None:
                continue
            for substring in substrings:
                if substring is None:
                    continue
                if str(substring).lower() in path_lowercase:
                    hits.append(group_key)
                    break  # one hit per group key is enough

        hits_unique = sorted(set(hits))
        matched_groups_col.append(hits_unique)
        matched_multi.append(len(hits_unique))
        if len(hits_unique) == 1:
            assigned.append(hits_unique[0])
        else:
            assigned.append(None)  # None for 0 or >1 hits; handle below

    df["group_ID"] = pd.Series(assigned, index=df.index).fillna("UNKNOWN")

    # Conflict detection at subject level: any subject mapped to >1 distinct non-UNKNOWN groups
    subject_groups = {}
    for i in range(len(df)):
        subject_id = df["BIDS_subject"].iloc[i]
        group_id = df["group_ID"].iloc[i]
        if pd.isna(subject_id):
            continue
        if group_id == "UNKNOWN":
            continue
        subject_groups.setdefault(subject_id, set()).add(group_id)

    conflicting_subjects = sorted([subject_id for subject_id, groups in subject_groups.items() if len(groups) > 1])

    # Also flag rows where a single path matched multiple groups (extra safety)
    row_conflict_subjects = df.loc[
        pd.Series(matched_multi, index=df.index) > 1,
        "BIDS_subject"
    ].dropna().unique().tolist()

    for subject_id in row_conflict_subjects:
        if subject_id not in conflicting_subjects:
            conflicting_subjects.append(subject_id)
    conflicting_subjects = sorted(conflicting_subjects)

    if conflicting_subjects:
        message = "[group_ID] Conflicting group matches for subjects: " + ", ".join(map(str, conflicting_subjects))
        if HARD_STOP:
            raise ValueError(
                message + ". Resolve overlapping substrings in 'group_identifiers' or adjust paths.")
        else:
            print("[warn]", message)
            print("[warn] Dropping all rows for these subjects due to ambiguous group assignment.")
            df = df[~df["BIDS_subject"].isin(conflicting_subjects)].reset_index(drop=True)

    # Clean up temp column
    if "_path_lc" in df.columns:
        df = df.drop(columns=["_path_lc"])

# 3) Summary counts per configured group key
counts = df["group_ID"].value_counts(dropna=False)
print("Number of 'group_ID' column assignments (number of files; not unique subject_IDs):")

group_keys = GROUP_MAPPINGS.keys() if GROUP_MAPPINGS else []
for group_key in group_keys:
    print(f"\t{group_key}: {int(counts.get(group_key, 0))}")

# Optional: report UNKNOWN count, if any
unknown_n = int(counts.get("UNKNOWN", 0))
if unknown_n:
    print(f"Number of UNKNOWN group_ID assignments: {unknown_n}")

In [ ]:
# =========================
# Build subject_ID from BIDS_subject (config-driven substring stripping)
# =========================

if "BIDS_subject" not in df.columns:
    raise KeyError("Column 'BIDS_subject' not found in df.")

# If not provided or empty -> no changes
if not STRIP_SUBJECTID_SUBSTRINGS:
    print("No redundant subject_ID substrings set in CONFIG; using full BIDS-derived subject identifiers.")
    df["subject_ID"] = df["BIDS_subject"]
else:
    # Make a working copy as strings (preserve original column)
    subject_series = df["BIDS_subject"].astype(str)

    # Track counts of affected identifiers per substring (count unique IDs touched)
    removal_counts = {}

    for substring in STRIP_SUBJECTID_SUBSTRINGS:
        if substring is None:
            continue
        substring = str(substring)  # ensure string

        # Identify which identifiers contain this exact (case-sensitive) substring
        has_substring = subject_series.str.contains(substring, na=False)

        # Count unique BIDS_subject identifiers affected (not occurrences)
        affected_ids = (
            df.loc[has_substring, "BIDS_subject"]
            .dropna()
            .astype(str)
            .unique())
        removal_counts[substring] = len(affected_ids)

        # Perform the replacement (remove all occurrences of the substring)
        subject_series = subject_series.str.replace(substring, "", regex=False)

    # Assign the result
    df["subject_ID"] = subject_series

    # Print concise summary for substrings that actually triggered removals
    any_removed = False
    for substring, count in removal_counts.items():
        if count == 0:
            print(
                f"No instances of substring: '{substring}' encountered in 'BIDS_subject' "
                "identifiers; check CONFIG value provided (Note: substrings are CASE-SENSITIVE).")
        if count > 0:
            any_removed = True
            print(f"Removed the substring: '{substring}' from {count} 'BIDS_subject' identifiers.")
    if not any_removed:
        print("No subject_ID substrings matched; 'subject_ID' remains identical to 'BIDS_subject'.")

Assemble final output:

In [ ]:
# =========================
# Pivot table to one row per subject_ID with per-session/per-type columns
# =========================

# --- Required columns sanity checks ---
required_columns = ["subject_ID", "group_ID", "data_type", "session_ID", "path"]
missing_required_columns = [column for column in required_columns if column not in df.columns]
if missing_required_columns:
    raise KeyError(f"Missing required columns in df: {missing_required_columns}")

# Ensure we have a 'pair_key' column (base path without extension)
def _make_pair_key(path_string: str) -> str:
    p = Path(path_string)
    name = p.name
    # Handle .nii.gz explicitly:
    if name.endswith(".nii.gz"):
        return str(p.with_name(name[:-len(".nii.gz")]))
    # Otherwise strip the last suffix (.nii, .fif, etc.)
    return str(p.with_suffix(""))
if "pair_key" not in df.columns:
    df["pair_key"] = df["path"].apply(_make_pair_key)

# --- Optional: refine session_ID using BIDS_run when multiple runs exist ---
if "BIDS_run" in df.columns:
    # Work only on rows where we actually have data_type, session_ID, and BIDS_run
    run_subset = df.dropna(subset=["data_type", "session_ID", "BIDS_run"])
    run_subset = run_subset.copy()
    run_subset["data_type"] = run_subset["data_type"].astype(str)
    run_subset["session_ID"] = run_subset["session_ID"].astype(str)
    run_subset["BIDS_run"] = run_subset["BIDS_run"].astype(str).str.strip()

    # Build mapping: (data_type, session_ID) -> sorted list of distinct run labels
    run_labels_by_type_and_session = {}
    if not run_subset.empty:
        for (current_data_type, current_session_id), group in run_subset.groupby(
                ["data_type", "session_ID"]):
            run_labels = (
                group["BIDS_run"]
                .dropna()
                .astype(str)
                .str.strip()
                .unique()
                .tolist())
            if len(run_labels) > 1:
                # Try to sort numerically if possible; fall back to string sort otherwise
                try:
                    run_labels_sorted = sorted(run_labels, key=lambda value: int(value))
                except ValueError:
                    run_labels_sorted = sorted(run_labels)
                run_labels_by_type_and_session[(current_data_type, current_session_id)] = run_labels_sorted

    # If any (data_type, session_ID) actually has multiple runs, update session_ID accordingly
    if run_labels_by_type_and_session:
        new_session_id_values = []
        for index, row in df.iterrows():
            original_session_id = row["session_ID"]
            if pd.isna(original_session_id) or pd.isna(row.get("data_type", np.nan)):
                new_session_id_values.append(original_session_id)
                continue

            # If BIDS_run is missing for this row, we leave session_ID unchanged
            bids_run_value = row.get("BIDS_run", np.nan)
            if pd.isna(bids_run_value):
                new_session_id_values.append(original_session_id)
                continue

            current_data_type = str(row["data_type"])
            current_session_id = str(original_session_id)
            key = (current_data_type, current_session_id)

            run_label_list = run_labels_by_type_and_session.get(key, None)
            if run_label_list is None:
                # This (data_type, session_ID) did not have multiple runs globally
                new_session_id_values.append(original_session_id)
                continue

            run_label_string = str(bids_run_value).strip()
            if run_label_string not in run_label_list:
                # Unexpected run label; leave unchanged
                new_session_id_values.append(original_session_id)
                continue

            # Map the run label to a 1-based index within its global ordering
            run_index = run_label_list.index(run_label_string) + 1
            new_session_id_values.append(f"{original_session_id}-{run_index}")

        df["session_ID"] = pd.Series(new_session_id_values, index=df.index)

# --- Sanity check: each subject_ID must map to a single group_ID ---
unique_group_counts_per_subject = (
    df[["subject_ID", "group_ID"]]
      .dropna(subset=["subject_ID"])
      .groupby("subject_ID")["group_ID"]
      .nunique(dropna=True))

conflicting_group_subject_ids = (
    unique_group_counts_per_subject[unique_group_counts_per_subject > 1]
    .index
    .tolist())

if conflicting_group_subject_ids:
    conflict_message = (
        "[group_ID] A subject_ID has multiple group_ID values: "
        + ", ".join(conflicting_group_subject_ids))
    if HARD_STOP:
        raise ValueError(conflict_message)
    else:
        print("[warn]", conflict_message)
        print("[warn] Dropping all rows for these conflicting subjects.")
        df = df[~df["subject_ID"].isin(conflicting_group_subject_ids)].reset_index(drop=True)

# --- Determine the unique data types present (programmatic, no assumptions) ---
unique_data_types = (
    pd.Series(df["data_type"])
      .dropna()
      .astype(str)
      .sort_values()
      .unique()
      .tolist())

# --- Build per-type session lists: only session_IDs that actually occur for each data_type ---
sessions_by_type = {}
for current_data_type in unique_data_types:
    subset = df[df["data_type"].astype(str) == current_data_type]
    session_values = (
        subset["session_ID"]
        .dropna()
        .astype(str)
        .sort_values()
        .unique()
        .tolist())
    sessions_by_type[current_data_type] = session_values

# --- Detect duplicates: more than one row for the same key ---
# Base key: (subject_ID, data_type, session_ID)
group_columns_for_duplicates = ["subject_ID", "data_type", "session_ID"]

# Only include BIDS_run if it exists AND has more than one distinct non-NaN value.
if "BIDS_run" in df.columns:
    non_null_runs = df["BIDS_run"].dropna()
    if non_null_runs.nunique() > 1:
        group_columns_for_duplicates.append("BIDS_run")

duplicate_key_counts = (
    df.dropna(subset=["subject_ID", "data_type", "session_ID"])
      .groupby(group_columns_for_duplicates)
      .size()
      .reset_index(name="row_count"))

subject_ids_with_duplicates = (
    duplicate_key_counts.loc[duplicate_key_counts["row_count"] > 1, "subject_ID"]
    .drop_duplicates()
    .tolist())

if subject_ids_with_duplicates:
    duplicates_message = (
        f"[dup] Detected {len(subject_ids_with_duplicates)} subject(s) with "
        f">1 file for the same key based on {group_columns_for_duplicates}.")
    if HARD_STOP:
        preview_subject_ids = ", ".join(subject_ids_with_duplicates[:10])
        raise ValueError(duplicates_message + f" Offenders include: {preview_subject_ids} ...")
    else:
        print("[warn]", duplicates_message)
        print("[warn] Example subjects (first 10):", ", ".join(subject_ids_with_duplicates[:10]))

# Prepare a quick lookup to flag subjects with any duplicates
has_duplicate_lookup = {
    subject_id: False for subject_id in df["subject_ID"].dropna().unique()}
for subject_id in subject_ids_with_duplicates:
    has_duplicate_lookup[subject_id] = True

# --- Build the pivoted per-subject table ---
unique_subject_ids_sorted = (
    pd.Series(df["subject_ID"])
      .dropna()
      .astype(str)
      .sort_values()
      .unique()
      .tolist())

# -------------------------
# Determine which data_type labels correspond to MRI-like vs MEG-like,
# driven entirely by config['datatype_mappings']:
# -------------------------
# Expectation: datatype_mappings is {<dtype_label>: [<BIDS_suffix>, ...], ...}
# Example: {"MRI": ["T1w"], "MEG": ["meg"]}

mri_like_labels = set()
meg_like_labels = set()

for dtype_label, suffix_list in (DATATYPE_MAPPINGS or {}).items():
    if not suffix_list:
        continue
    suffixes_normalized = {str(s).strip() for s in suffix_list if s is not None}

    # Heuristic: MRI-like if it includes T1w (or other anat suffixes you may add later)
    if "T1w" in suffixes_normalized:
        mri_like_labels.add(str(dtype_label))

    # Heuristic: MEG-like if it includes meg
    if "meg" in suffixes_normalized:
        meg_like_labels.add(str(dtype_label))

# If config is unconventional, fall back to the literal labels if present
observed_dtype_labels = set(df["data_type"].dropna().astype(str).unique().tolist())
if not mri_like_labels and "MRI" in observed_dtype_labels:
    mri_like_labels.add("MRI")
if not meg_like_labels and "MEG" in observed_dtype_labels:
    meg_like_labels.add("MEG")

# Warn if still empty (don’t hard-stop; just make the failure mode explicit)
if not mri_like_labels:
    print("[warn] Could not infer MRI-like data_type label(s) from datatype_mappings (looked for suffix 'T1w').")
if not meg_like_labels:
    print("[warn] Could not infer MEG-like data_type label(s) from datatype_mappings (looked for suffix 'meg').")

pivoted_subject_rows = []
for subject_id in unique_subject_ids_sorted:
    subject_slice = df[df["subject_ID"] == subject_id]

    # Single group_ID per subject (ensured by earlier check / drop)
    subject_group_values = (
        subject_slice["group_ID"]
        .dropna()
        .astype(str)
        .unique()
        .tolist())
    subject_group_id = subject_group_values[0] if subject_group_values else "UNKNOWN"

    # Set Boolean coverage flags (MRI + MEG):

    # derived from config-driven datatype_mappings:
    subject_dtype_series = subject_slice["data_type"].dropna().astype(str)

    has_mri_flag = bool(subject_dtype_series.isin(mri_like_labels).any())
    has_meg_flag = bool(subject_dtype_series.isin(meg_like_labels).any())

    num_mri_files = int(subject_dtype_series.isin(mri_like_labels).sum())
    num_meg_files = int(subject_dtype_series.isin(meg_like_labels).sum())

    # Initialize the base record
    subject_record = {
        "subject_ID": subject_id,
        "group_ID": subject_group_id,
        "has_MRI": has_mri_flag,
        "has_MEG": has_meg_flag,
        "n_MRI": num_mri_files,
        "n_MEG": num_meg_files,
        "n_good_MRIs": np.nan,
        "n_good_MEGs": np.nan,
        "first_good_MRI": np.nan,
        "first_good_MEG": np.nan,
        "HAS_DUPLICATE": has_duplicate_lookup.get(subject_id, False)}

    # Dynamic filename/QC columns for each (data_type, session_ID) that actually exists for that data_type
    for current_data_type in unique_data_types:
        session_list = sessions_by_type.get(current_data_type, [])
        for current_session_id in session_list:
            matching_subset = subject_slice[
                (subject_slice["data_type"].astype(str) == current_data_type) &
                (subject_slice["session_ID"].astype(str) == current_session_id)]

            # Base filename from pair_key (strip directory), or np.nan if not present
            if matching_subset.empty:
                base_filename = np.nan
            else:
                sorted_subset = matching_subset.sort_values("pair_key", kind="stable")
                base_filename = Path(sorted_subset["pair_key"].iloc[0]).name

            filename_column_name = f"{current_data_type}_{current_session_id}_filename"
            qc_summary_column_name = f"{current_data_type}_{current_session_id}_QC_summary"
            qc_comment_column_name = f"{current_data_type}_{current_session_id}_QC_comment"

            subject_record[filename_column_name] = base_filename
            subject_record[qc_summary_column_name] = np.nan
            subject_record[qc_comment_column_name] = np.nan

    pivoted_subject_rows.append(subject_record)

pivot_df = pd.DataFrame(pivoted_subject_rows)

# --- Final column ordering: requested core columns first, then dynamic filename/QC columns ---
core_column_order = [
    "subject_ID",
    "group_ID",
    "has_MRI",
    "has_MEG",
    "n_MRI",
    "n_MEG",
    "n_good_MRIs",
    "n_good_MEGs",
    "first_good_MRI",
    "first_good_MEG",
    "HAS_DUPLICATE"]

dynamic_columns_in_order = []
for current_data_type in unique_data_types:
    session_list = sessions_by_type.get(current_data_type, [])
    for current_session_id in session_list:
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_filename")
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_QC_summary")
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_QC_comment")

all_columns_now = pivot_df.columns.tolist()
final_column_order = (
    [column for column in core_column_order if column in all_columns_now]
    + [column for column in dynamic_columns_in_order if column in all_columns_now])

pivot_df = pivot_df[final_column_order]

print(f"[result] pivoted_df shape: {pivot_df.shape[0]} rows × {pivot_df.shape[1]} cols")
duplicate_subject_total = int(pivot_df["HAS_DUPLICATE"].sum())
if duplicate_subject_total:
    print(
        f"[warn] {duplicate_subject_total} subject(s) have duplicate files for at least one "
        f"combination of {group_columns_for_duplicates}.")

In [ ]:
pivot_df.sample(7)

--------
#### Final save / export:

In [ ]:
# Convert path string from config.yaml into valid Path object:
base_output_directory = Path(base_output_directory)

output_file_path = base_output_directory / "master_data_catalogue.csv"

# Export the dataframe to .csv:
pivot_df.to_csv(output_file_path, index=False)

print(f"[write] master_data_catalogue.csv → {output_file_path.resolve()}")
print(f"[result] Exported {pivot_df.shape[0]} rows × {pivot_df.shape[1]} columns.")